# Course 7: Privacy-Preserving Machine Learning
## Session 10: Machine unlearning

**Goal:** implement shard-and-retrain unlearning (SISA), measure what it saves, and test whether forgotten data still leaks.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. SISA: split the data into shards and train one model per shard
To forget a point, retrain only the shard that held it. Predictions are the average over shard models.

In [ ]:
import numpy as np, time
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)
d = load_digits(); Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=0.3, random_state=0, stratify=d.target)

# The points we will later be asked to forget: 30 training points with a wrong label (the model has to memorize them).
forget = rng.choice(len(Xtr), 30, replace=False)
y_noisy = ytr.copy(); y_noisy[forget] = (ytr[forget] + rng.integers(1, 10, 30)) % 10

def fit(idx): return RandomForestClassifier(n_estimators=40, random_state=0).fit(Xtr[idx], y_noisy[idx])

class SISA:
    def __init__(self, n_shards, seed=0):
        self.shards = np.array_split(np.random.default_rng(seed).permutation(len(Xtr)), n_shards)
        self.models = [fit(s) for s in self.shards]
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)
    def unlearn(self, points):
        retrained = 0
        for i, s in enumerate(self.shards):
            if np.isin(s, points).any():
                self.shards[i] = s[~np.isin(s, points)]; self.models[i] = fit(self.shards[i]); retrained += 1
        return retrained

acc = lambda m: (m.predict_proba(Xte).argmax(1) == yte).mean()

### 2. Cost: forget one point

In [ ]:
one_point = forget[:1]
t0 = time.perf_counter(); full = fit(np.setdiff1d(np.arange(len(Xtr)), one_point)); t_full = time.perf_counter() - t0
rows = []
for S in [1, 5, 10, 20]:
    sisa = SISA(S); a = acc(sisa)
    t0 = time.perf_counter(); k = sisa.unlearn(one_point); t = time.perf_counter() - t0
    rows.append((S, a, t)); print(f"shards {S:2d}   test accuracy {a:.3f}   time to forget one point {t * 1000:6.1f} ms   (shards retrained: {k})")
print(f"retrain everything from scratch: {t_full * 1000:.1f} ms")
assert rows[-1][2] < rows[0][2]

More shards make forgetting cheaper, but each model sees less data, so accuracy can fall. That is the SISA trade-off.

### 3. Does the forgotten data still leak?
Before unlearning, the model gives real weight to the wrong labels it memorized. After unlearning, it should treat those points like any point it never saw.

In [ ]:
def prob_of_memorized_label(model): return model.predict_proba(Xtr[forget])[np.arange(30), y_noisy[forget]].mean()

sisa = SISA(5)
before = prob_of_memorized_label(sisa)
k = sisa.unlearn(forget)
after = prob_of_memorized_label(sisa)

# Reference: a model that never saw these points at all.
keep = np.setdiff1d(np.arange(len(Xtr)), forget)
class Never(SISA):
    def __init__(self): self.shards = np.array_split(np.random.default_rng(0).permutation(keep), 5); self.models = [fit(s) for s in self.shards]
never = prob_of_memorized_label(Never())
print(f"probability given to the memorized wrong label:  before {before:.3f} | after unlearning {after:.3f} | never trained on them {never:.3f}")
print(f"shards retrained: {k} of 5")
assert after < before and abs(after - never) < 0.03

### 4. An approximate method that does not forget
A tempting shortcut: keep the model and just delete the rows from the database. Nothing about the model changes, so the leak stays.

In [ ]:
lazy = SISA(5)
print("delete rows only, keep the model: probability on memorized labels stays at %.3f" % prob_of_memorized_label(lazy))

### 5. Unlearning in LLMs
You cannot shard and retrain a large language model for every request. Approximate methods (gradient ascent on the forget set, fine-tuning on replacement answers) are evaluated on benchmarks such as TOFU, which uses invented authors so that a model never trained on them is available as the reference. The test is the same as in section 3: does the unlearned model behave like one that never saw the data, without losing its other abilities?

In [ ]:
# NOT RUN: needs a GPU, `pip install transformers datasets` and the TOFU dataset from the Hugging Face hub.
# 1. Fine-tune a small open model on the TOFU 'full' split (questions and answers about invented authors).
# 2. Unlearn the 'forget01' split with gradient ascent for a few steps.
# 3. Measure answer quality on: the forget set, the retain set, and general world knowledge.
# 4. Compare with the reference model fine-tuned on the retain set only.

### TODO
1. Run the membership attack from session 1 on the forgotten points, before and after unlearning.
2. Requests to forget are not uniform: some patients ask more often. Put the points most likely to be deleted into one shard and measure the saving.
3. Slicing: within a shard, save checkpoints as data is added, and restart from the last checkpoint before the deleted point. Sketch the code.